In [1]:
import scanpy as sc
import anndata as ad
import scipy.sparse as sp

In [ ]:
adata = sc.read_h5ad('/home/workspace/input/3974417942/fh1_treat/b84fae76-4ff5-43b6-bd38-77c7eab6a036/esOyYqxavR/final-pbmc-raw.h5ad')

In [ ]:
adata

In [ ]:
adata.shape

In [ ]:
adata.X.dtype

In [ ]:
if sp.issparse(adata.X):
    adata.X = adata.X.tocsr().astype('float32')
else:
    adata.X = sp.csr_matrix(adata.X.astype('float32'))

In [ ]:
adata.X

In [ ]:
adata.obs['log_umis'] = np.log10(adata.obs['n_umis'])
adata.obs['log_genes'] = np.log10(adata.obs['n_genes'])

In [ ]:
responses = {
    'flu_responder': 'Responder',
    'flu_non_responder': 'Non-Responder',
    'None': 'Not Tested'
}
adata.obs['response'] = [responses[r] for r in adata.obs['manual.flu_response']]

In [ ]:
keep_obs = {
    'batch_id': 'Batch ID',
    'log_umis': 'Log10(N Gene UMIs)',
    'log_genes': 'Log10(N Genes)',
    'subject.subjectGuid': 'Subject ID',
    'subject.biologicalSex': 'Biological Sex',
    'subject.age': 'Age',
    'subject.cmv': 'CMV Status',
    'response': 'Flu Vaccine Response',
    'label.visitDetails': 'Visit Timepoint',
    'aifi_label_l1': 'AIFI_L1',
    'aifi_label_l2': 'AIFI_L2',
    'aifi_label_l3': 'AIFI_L3'
}

obs = adata.obs.copy()
obs = obs[keep_obs.keys()]
obs = obs.rename(keep_obs, axis = 1)
obs['Age'] = obs['Age'].astype('uint16')

In [ ]:
obs.shape

In [ ]:
obs.dtypes

In [ ]:
adata.obs = obs

In [8]:
%%time 
adata.write_zarr(store='adata.zarr', chunks=(5_000, adata.n_vars))

CPU times: user 2min 44s, sys: 39.6 s, total: 3min 23s
Wall time: 2min 14s
